# State

In [26]:
from typing import TypedDict

class State(TypedDict):
    operation: str
    processed_value: int

obj = State({'first_value':1,"second_value":2,"answer":3})
obj['first_value']

1

In [ ]:
# from pydantic import BaseModel

# class State(BaseModel):
#     first_value: int
#     second_value: int
#     answer: int

# Graph

In [27]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

# Node

In [28]:
def additional_node(state: State):
    value = state.get('processed_value')
    out = value+2
    return {
        'processed_value':out
    }

def subraction_node(state: State):
    value = state.get('processed_value')
    out = value-2
    return {
        'processed_value':out
    }

def multiply_node(state: State):
    value = state.get('processed_value')
    out = value*2
    return {
        'processed_value':out
    }

graph.add_node("addition",additional_node)
graph.add_node("sub",subraction_node)
graph.add_node("multiply", multiply_node)
    

# Edge

In [ ]:
graph.add_edge(START, "addition")
graph.add_edge("addition","sub")
graph.add_edge("sub","multiply")
graph.add_edge("multiply",END)

# COMPILE

In [23]:
agent = graph.compile()

# Invoking

In [24]:
response = agent.invoke({'processed_value': 4})
print(response.get('processed_value'))

8


In [25]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

# Conditional edges

In [29]:
def router_function(state: State):
    operation = state.get('operation')
    if operation == 'add':
        return 'add'
    elif operation == 'sub':
        return 'sub'
    elif operation == 'mul':
        return 'mul'

graph.add_conditional_edges(
    START,
    router_function,
    {
        'add':'addition',
        'sub':'sub',
        'mul':'multiply'
    }
)

graph.add_edge('addition',END)
graph.add_edge('sub',END)
graph.add_edge('multiply',END)

In [32]:
agent = graph.compile()

response = agent.invoke({'operation':'mul','processed_value':6})
print(response['processed_value'])

12


In [33]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

In [ ]:
[
    {'role':'ai','content':'hello, how can i help you today'}
]

# AI agent

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

# response = llm.invoke('hii')

def chat_llm(state: State):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

graph = StateGraph(State)

graph.add_node('llm_node', chat_llm)

graph.add_edge(START, 'llm_node')
graph.add_edge('llm_node', END)

agent = graph.compile()

response = agent.invoke({'messages':'hi'})
print(response['messages'][-1].content)

response = agent.invoke({'messages':'I am aravind'})
print(response['messages'][-1].content)

response = agent.invoke({'messages':'what is my name as i told before'})
print(response['messages'][-1].content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hi there! How can I help you today?
Hello Aravind! It's good to meet you.

How can I help you today?
As an AI, I don't have a memory of past conversations or personal information like your name. I don't store any details about you.

If you'd like me to refer to you by a specific name during this interaction, please feel free to tell me what you'd like to be called!


# Checkpointer

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph = StateGraph(State)

memory = MemorySaver()

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

def chat_llm(state: State):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

graph = StateGraph(State)

graph.add_node('llm_node', chat_llm)

graph.add_edge(START, 'llm_node')
graph.add_edge('llm_node', END)

agent = graph.compile(checkpointer = memory)

config = {'configurable':{'thread_id':'id_1'}}

response = agent.invoke({'messages':'hi'}, config=config)
print(response['messages'][-1].content)

print("--------------------------------------------------")

response = agent.invoke({'messages':'I am aravind'}, config=config)
print(response['messages'][-1].content)

print("--------------------------------------------------")
response = agent.invoke({'messages':'what is my name as i told before'}, config=config)
print(response['messages'][-1].content)

Hi there! How can I help you today?
--------------------------------------------------
Nice to meet you, Aravind! I'm a large language model, I don't have a name like humans do, but you can call me Assistant if you like.

How can I help you today, Aravind? Or what would you like to talk about?
--------------------------------------------------
Your name is Aravind!


In [5]:
response = agent.invoke({'messages':'what is my name as i told before'})
print(response['messages'][-1].content)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [3]:
response = agent.invoke({'messages':'what is my name as i told before'}, config=config)
print(response['messages'][-1].content)

Your name is Aravind.


In [4]:
config = {'configurable':{'thread_id':'id_2'}}
response = agent.invoke({'messages':'what is my name as i told before'}, config=config)
print(response['messages'][-1].content)

As an AI, I don't have memory of past conversations or personal information about you. I don't know your name unless you tell it to me during our current interaction.

So, no, you haven't told me your name before (in a way that I would remember it).


# Tool

In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
import uuid

@tool
def uniqueId_generator():
    """This function will create a unique id, so call this function when user request to create a uuid."""
    return str(uuid.uuid4())

@tool
def add_numbers(a:int, b:int) -> int:
    """This tool will add the given input
    
    Args:
    a (int): get the first number from the query
    b (int): get the second number to add from the query

    Return:
    out (int): summation of the input
    """
    return a+b

tools = [uniqueId_generator, add_numbers]


class State(TypedDict):
    messages: Annotated[list, add_messages]

graph = StateGraph(State)

memory = MemorySaver()

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
llm = llm.bind_tools(tools)

def chat_llm(state: State):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

graph = StateGraph(State)

graph.add_node('llm_node', chat_llm)
graph.add_node('tools', ToolNode(tools))

graph.add_edge(START, 'llm_node')
graph.add_conditional_edges(
    'llm_node',
    tools_condition
)
graph.add_edge('tools', 'llm_node')

agent = graph.compile(checkpointer = memory)

config = {'configurable':{'thread_id':'id_4'}}

In [14]:
response = agent.invoke({'messages':'hi'}, config=config)
print(response['messages'][-1].content)

Hello! I'm a calculator bot, I can add numbers for you, or generate a unique ID. How can I help?


In [16]:
response = agent.invoke({'messages':'could you please generate a unique for me'}, config=config)

for i in response['messages']:
    print(i)
    print("************************")


print("========================================")
print(response['messages'][-1].content)

content='hi' additional_kwargs={} response_metadata={} id='866027f5-87e8-463e-b426-44d733e07312'
************************
content="Hello! I'm a calculator bot, I can add numbers for you, or generate a unique ID. How can I help?" additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1166a-62ff-73b2-ae79-cdb60dd336a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 140, 'output_tokens': 27, 'total_tokens': 167, 'input_token_details': {'cache_read': 0}}
************************
content='could you please generate a unique for me' additional_kwargs={} response_metadata={} id='cc2ed027-70e6-4312-a079-b2e3a0efc0e4'
************************
content='' additional_kwargs={'function_call': {'name': 'uniqueId_generator', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'e29060cf-3a80-4b0f-8874-7f545d5ce6be': 'CtEBAWkUfRNZf1pQZsyP3XfvgBDrl3TTqM

In [17]:
response = agent.invoke({'messages':'what is the output of 2+2'}, config=config)

for i in response['messages']:
    print(i)
    print("************************")


print("========================================")
print(response['messages'][-1].content)

content='hi' additional_kwargs={} response_metadata={} id='866027f5-87e8-463e-b426-44d733e07312'
************************
content="Hello! I'm a calculator bot, I can add numbers for you, or generate a unique ID. How can I help?" additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1166a-62ff-73b2-ae79-cdb60dd336a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 140, 'output_tokens': 27, 'total_tokens': 167, 'input_token_details': {'cache_read': 0}}
************************
content='could you please generate a unique for me' additional_kwargs={} response_metadata={} id='cc2ed027-70e6-4312-a079-b2e3a0efc0e4'
************************
content='' additional_kwargs={'function_call': {'name': 'uniqueId_generator', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'e29060cf-3a80-4b0f-8874-7f545d5ce6be': 'CtEBAWkUfRNZf1pQZsyP3XfvgBDrl3TTqM